In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os, sys
sys.path.append('/Users/walu/icecube/bd_ppros/')
from Event import Event
from Photon import Photon
os.chdir('/Users/walu/icecube/doumeki_analysis/')
import copy
import h5py
import numpy as np
from scipy.interpolate import interp1d

#to calculate cross section weight
from asteria.interactions import InvBetaPar, InvBetaTab
from snewpy.neutrino import Flavor
import astropy.units as u

In [ ]:
master_df = pd.read_csv('mdom_ibd_combined_52550199_nu_energy_flat_spectrum.csv')

In [ ]:
HC = 1241  # eV·nm

events = []

for run_id, group in master_df.groupby('run_id'):
    # --- Event-level attributes ---
    enu = group['enu'].iloc[0]  # neutrino energy for this run
    ee = group['ee'].iloc[0]    # positron energy for this run
    x, y, z = group[['vertex_x', 'vertex_y', 'vertex_z']].iloc[0]
    time = group['hit_time'].min()  # you can use .mean() or .iloc[0] if preferred
    runID = run_id

    # --- Build photon list for this event ---
    photons = []
    for _, row in group.iterrows():
        energy_eV = row['photon_energy']  # assuming photon_energy is already in eV
        wavelength_m = (HC / energy_eV)  # convert nm
        pmt = row['pmt_hit']
        photon = Photon(wavelength=wavelength_m, pmt=pmt, runID=runID)
        photons.append(photon)

    # --- Create event and attach photons ---
    event = Event(enu=enu, ee=ee, x=x, y=y, z=z, runID=runID, time=time, weight=1.0)
    event.add_photon_list(photons)
    event.calculate_multiplicity()

    events.append(event)

del master_df

In [ ]:
def build_distribution(h5_path, mass_hierarchy="nmo", time_bin_width=0.01, energy_bin_width=2.0):
    """
    Build normalized time and energy distributions from a supernova neutrino model HDF5 file.

    Parameters
    ----------
    h5_path : str
        Path to the .h5 file (e.g., '.../Tamborra2014_27.0Msun_dir1_NU_E_BAR_with_osc.h5')
    mass_hierarchy : str, optional
        One of {'no_osc', 'nmo', 'imo'}. Determines which spectra to use.
    time_bin_width : float, optional
        Desired time bin width in seconds.
    energy_bin_width : float, optional
        Desired energy bin width in MeV.

    Returns
    -------
    result : dict
        {
            "time_grid": 1D array of new time centers,
            "energy_grid": 1D array of new energy centers (rebinned for spectra),
            "spectra_rebinned": 2D array [energy, time],
            "flux_rebinned": 1D total flux vs time,
            "time_distribution": normalized 1D PDF over time (sum = 1),
        }
    """

    # ---------------------------
    # 1. Load the model data
    # ---------------------------
    with h5py.File(h5_path, "r") as f:
        group_name = list(f["models"].keys())[0]
        g = f[f"models/{group_name}"]

        time = g["time"][:]              # seconds
        energy = g["energy"][:]          # MeV
        flux = g["flux"][:]              # 1D flux (vs time)
        spectra = g["spectra"][mass_hierarchy][:].T  # shape: (energy, time)

    # ---------------------------
    # 2. Re-bin time
    # ---------------------------
    t_min, t_max = time.min(), time.max()
    time_edges = np.arange(t_min, t_max + time_bin_width, time_bin_width)
    time_centers = 0.5 * (time_edges[:-1] + time_edges[1:])

    # Interpolate spectra onto new time centers
    interp_spec_time = interp1d(time, spectra, kind="linear", axis=1, fill_value="extrapolate")
    spectra_time_rebinned = interp_spec_time(time_centers)  # shape (E, new_T)

    # Also interpolate flux to same time centers
    interp_flux = interp1d(time, flux, kind="linear", fill_value="extrapolate")
    flux_rebinned = interp_flux(time_centers)

    # ---------------------------
    # 3. Re-bin spectra in energy
    # ---------------------------
    E_min, E_max = energy.min(), energy.max()
    E_edges = np.arange(E_min, E_max + energy_bin_width, energy_bin_width)
    E_centers = 0.5 * (E_edges[:-1] + E_edges[1:])

    # Interpolate spectrum to coarser energy bins
    interp_spec_energy = interp1d(energy, spectra_time_rebinned, kind="linear", axis=0, fill_value="extrapolate")
    spectra_rebinned = interp_spec_energy(E_centers)  # shape: (new_E, new_T)

    # ---------------------------
    # 4. Build normalized time distribution
    # ---------------------------
    flux_integrated = np.trapz(spectra_rebinned, E_centers, axis=0)
    time_distribution = flux_integrated / np.sum(flux_integrated)

    # ---------------------------
    # 5. Return results
    # ---------------------------
    return {
        "time_grid": time_centers,
        "energy_grid": E_centers,
        "spectra_rebinned": spectra_rebinned,
        "flux_rebinned": flux_rebinned,
        "time_distribution": time_distribution,
    }


In [ ]:
snewpy_distribution = build_distribution('/Users/walu/icecube/energy_constraints/snecc/data/snewpy_model_data/sukhbold_ls220/sukhbold_27.00Msun_NU_E_BAR_with_osc.h5', mass_hierarchy='no_osc')

In [ ]:
def sample_time(events, snewpy_distribution):

    E_grid = snewpy_distribution['energy_grid']
    spectra = snewpy_distribution['spectra_rebinned']  # shape: (E, time)
    time_grid = snewpy_distribution['time_grid']
    flux_weight = snewpy_distribution['flux_rebinned']

    #Calculating avg energy for each time bin
    avg_energy = []
    for j in range(spectra.shape[1]):
        num = np.trapz(E_grid * spectra[:, j], E_grid)
        denom = np.trapz(spectra[:, j], E_grid)
        if np.isnan(num) or np.isnan(denom) or denom == 0:
            avg_energy.append(0.0)
        else:
            avg_energy.append(num / denom)
    avg_energy = np.array(avg_energy)

    # Sample time from flux_weight vs time distribution
    interp_flux = interp1d(time_grid, flux_weight, kind="linear", fill_value="extrapolate")
    flux_vals = interp_flux(time_grid)

    # Make sure no negative or nan values
    flux_vals = np.nan_to_num(flux_vals, nan=0.0)
    flux_vals = np.clip(flux_vals, a_min=0, a_max=None)

    # Divide safely: replace NaN, inf, or div-by-zero with 0
    with np.errstate(divide='ignore', invalid='ignore'):
        energy_averaged_flux = np.divide(flux_vals, avg_energy)
        energy_averaged_flux[~np.isfinite(energy_averaged_flux)] = 0.0  # set nan, inf to 0

    
    flux_pdf = energy_averaged_flux / np.sum(energy_averaged_flux)  # normalize to sum = 1

    #loading the sampled time for each event
    sampled_times = np.random.choice(time_grid, size = len(events), p = flux_pdf)
    for i, event in enumerate(events):
        event.time = sampled_times[i]

    return events

def ccsn_spectrum_weight(events, snewpy_distribution):

    """
    Applies the first weight
    """
    E_grid = snewpy_distribution['energy_grid']
    spectra = snewpy_distribution['spectra_rebinned']  # shape: (E, time)
    time_grid = snewpy_distribution['time_grid']    

    #normalizing the spectra
    for j in range(spectra.shape[1]):
        spectra[:, j] =  spectra[:, j]/ np.trapezoid(spectra[:, j], E_grid)

    #weighting the events with the spectrum at the sampled time
    weighted_events = []
    for i, event in enumerate(events):
        #finding closest time to the sampled time
        time_idx = np.argmin(np.abs(time_grid - event.time))
        spectrum = spectra[:, time_idx] #getting the spectrum at the sampled time
        interp_spec = lambda E: np.interp(E, E_grid, spectrum, left=0.0, right=0.0)
        weight = interp_spec(event.enu)
        #adding none wight warning
        event.weight = weight
        weighted_events.append(event)
        
    return weighted_events

def cross_section_weight(events):
    ibd = InvBetaPar()
    l = (40 * u.m).to(u.cm).value
    n_target =  6.023e23 * (2 / 18)
    for event in events:
        xs = ibd.cross_section(Flavor.NU_E_BAR, event.enu * u.MeV)
        event.weight = event.weight * xs.value * n_target * l
    return events


def flux_weight(events, snewpy_distribution, Ngen=2362019, distance = 10 * u.kpc, radius = 20 * u.m):
    E_grid = (snewpy_distribution['energy_grid'] * u.MeV).value
    spectra = snewpy_distribution['spectra_rebinned']  
    time_grid = (snewpy_distribution['time_grid'] * u.s).value
    luminosity = ((snewpy_distribution['flux_rebinned'] * (u.erg /u.s)).to(u.MeV / u.s)).value * 1e51

    #Calculating avg energy for each time bin
    avg_energy = []
    for j in range(spectra.shape[1]):
        num = np.trapz(E_grid * spectra[:, j], E_grid)
        denom = np.trapz(spectra[:, j], E_grid)
        if np.isnan(num) or np.isnan(denom) or denom == 0:
            avg_energy.append(0.0)
        else:
            avg_energy.append(num / denom)
    avg_energy = np.array(avg_energy)

    # Sample time from flux_weight vs time distribution
    interp_flux = interp1d(time_grid, luminosity, kind="linear", fill_value="extrapolate")
    flux_vals = interp_flux(time_grid)

    # Make sure no negative or nan values
    flux_vals = np.nan_to_num(flux_vals, nan=0.0)
    flux_vals = np.clip(flux_vals, a_min=0, a_max=None)

    # Divide safely: replace NaN, inf, or div-by-zero with 0
    with np.errstate(divide='ignore', invalid='ignore'):
        energy_averaged_flux = np.divide(flux_vals, avg_energy)
        energy_averaged_flux[~np.isfinite(energy_averaged_flux)] = 0.0  # set nan, inf to 0

    distance = distance.to(u.m).value
    #integrating total flux over time grid
    flux_weight = (1/Ngen) * (radius.to(u.m).value ** 2 / (distance ** 2)) * np.trapz(energy_averaged_flux, time_grid)

    for event in events:
        event.weight = event.weight * flux_weight
    return events

def eff_vol_weight(events, abs_file_path, absorption_sim, N_modules, max_multiplicity, multiplicity_params):
    """
    Calculate W_eff(m) for multiplicities up to max_multiplicity.
    """
    with open(abs_file_path, 'r') as f:
        absorption_lengths = [float(line.strip()) for line in f if line.strip()]

    weights = {}

    for m in range(1, max_multiplicity + 1):
        m_key = m if m in multiplicity_params else 6
        b, c = multiplicity_params[m_key]

        # Mean effective volume across all absorption lengths
        V_eff_all = [b * (1.0 / L) + c for L in absorption_lengths]
        V_eff_mean = np.mean(V_eff_all)

        # Effective volume at provided absorption length
        V_eff_sim = b * (1.0 / absorption_sim) + c

        # Weight for this multiplicity
        W_eff = N_modules * (V_eff_mean / V_eff_sim)
        weights[m] = W_eff

    #  Apply multiplicative weighting
    for e in events:
        if hasattr(e, 'multiplicity') and e.multiplicity is not None:
            m = e.multiplicity
            W_eff = weights[m] if m in weights else weights[max(weights)]

            
            e.weight = e.weight * W_eff

    return events

def QE_weights(events, qe_file_path=None, min_multiplicity = 1, max_multiplicity = 24):
    wavelength, qe = np.loadtxt(qe_file_path, dtype=float, unpack=True)
    wavelength_pdf = qe  #QE should not be normalized, it does not have to be
    interp_qe = interp1d(wavelength, wavelength_pdf, kind="linear", fill_value=0.0, bounds_error=False)
    for event in events:
        photons = event.get_photons()
        p_list = np.zeros(len(photons))
        for i, photon in enumerate(photons):
            p_list[i] = interp_qe(photon.wavelength)
        total_p = np.mean(p_list)
        event.weight = event.weight * total_p  



    return events


In [ ]:
# Suppose you already have:
# events = [...]  # your list of Event objects
# ibd = InvBetaPar()  # optional, not needed for this function
events = sample_time(events, snewpy_distribution)
events = ccsn_spectrum_weight(events, snewpy_distribution)
events = cross_section_weight(events)
events = flux_weight(events, snewpy_distribution)   
events = QE_weights(events, qe_file_path='qe_data.txt')
multiplicity_params = {
    1: (15.1843, 0.000),
    2: (0.1352, 47.005),
    3: (0.0306, 16.170),
    4: (0.0095, 8.680),
    5: (0.0039, 5.105),
    6: (0.0026, 2.909),  # used for m >= 6
}

#spectra_dir = '/Users/walu/icecube/software_archive/snewpy/models/Sukhbold_2015/sukhbold_spectra/'
weighted_events = eff_vol_weight(
    events=events, 
    abs_file_path='absorption.txt',
    absorption_sim=227.2,
    N_modules=10000,
    max_multiplicity=24,
    multiplicity_params=multiplicity_params
    )

"""
weighted_events = QE_weights(
    events=weighted_events, 
    qe_file_path='qe_data.txt'
)
"""

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def multiplicity_histogram(weighted_events, model_name, savepath=None):
    """
    Plot a multiplicity histogram for weighted Event objects.

    Parameters
    ----------
    weighted_events : list of Event
        List of weighted/surviving Event objects, each with attributes
        'multiplicity' and 'weight'.
    model_name : str
        Name of the SN model or dataset.
    savepath : str, optional
        If provided, saves the histogram to this path instead of showing it.
    """
    # Extract multiplicities and weights
    multiplicities = np.array([e.multiplicity for e in weighted_events if e.multiplicity is not None])
    weights = np.array([e.weight for e in weighted_events if e.multiplicity is not None])
    #weights = np.nan_to_num(weights, nan=0.0)

    if len(multiplicities) == 0:
        print("⚠️ No valid multiplicity values found.")
        return

    # Histogram with weights
    bins = np.arange(min(multiplicities), max(multiplicities) + 2) - 0.5

    plt.figure(figsize=(7, 5))
    plt.hist(
        multiplicities,
        bins=bins,
        weights=weights,  # <-- weighted histogram
        edgecolor='black',
        alpha=0.75,
        color='royalblue',
    )

    plt.yscale('log')
    plt.xlabel("Multiplicity (unique PMTs hit)", fontsize=13)
    plt.ylabel("Weighted number of events", fontsize=13)
    plt.title(f"Multiplicity Distribution – {model_name}", fontsize=14)
    plt.grid(True, linestyle='--', alpha=0.4, which='both')
    plt.xlim([0, 15.5])
    plt.ylim([1e-1, None])

    if savepath:
        plt.tight_layout()
        plt.savefig(savepath, dpi=300)
        plt.close()
        print(f"✅ Weighted multiplicity histogram saved to {savepath}")
    else:
        plt.show()


In [ ]:
multiplicity_histogram(weighted_events, model_name = 'sulhbold_2015_no_osc')